In [ ]:
from pathlib import Path
import subprocess, os, json, shutil
ROOT=Path('/content/agent-evals-fresh-20261014')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/yuvasree023/agent-evals.git',str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','--quiet','origin'],check=True)
    subprocess.run(['git','-C',str(ROOT),'reset','--hard','--quiet','origin/main'],check=True)
subprocess.run(['git','checkout','--detach','3ad211c78086874cde91307abeeaa6ab9c34cfa3'],cwd=ROOT,check=True)
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip())
# Standard library only; no dependency installation needed.
subprocess.run(['python3','-m','unittest','discover','-s','tests'],cwd=ROOT,check=True)
subprocess.run(['python3','run.py','--validate'],cwd=ROOT,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip() == '3ad211c78086874cde91307abeeaa6ab9c34cfa3'
from google.colab import userdata
os.environ['GROQ_API_KEY']=userdata.get('GROQ_API_KEY')
# Use Groq's documented official SDK with its default client identity.
# No browser identity, custom User-Agent, proxy, or signature override.
import sys
subprocess.run([sys.executable,'-m','pip','install','groq'],check=True)
import sys, importlib
sys.path.insert(0,str(ROOT))
path=ROOT/'providers.py'
source=path.read_text()
marker='    body = json.dumps(payload).encode()'
replacement='''    if provider == "groq":
        from groq import Groq, APIStatusError, APIConnectionError
        _pace(provider)
        try:
            client = Groq(api_key=_key(provider), max_retries=2, timeout=120)
            data = client.chat.completions.create(**payload)
            return 200, data.model_dump(), {}
        except APIStatusError as exc:
            if exc.status_code == 429 and (exc.response.headers.get("x-ratelimit-remaining-requests") == "0" or "tokens per day" in exc.response.text.lower() or "daily" in exc.response.text.lower()):
                raise DailyQuotaExhausted("groq daily request quota exhausted")
            raise ProviderError(f"groq SDK HTTP {exc.status_code}: {exc.response.text[:500]}")
        except APIConnectionError:
            raise ProviderError("groq SDK connection failed")
    body = json.dumps(payload).encode()'''
if marker not in source:
    raise RuntimeError('Provider transport marker missing')
# Ephemeral Colab checkout only. Provenance is printed, no secret values.
if 'from groq import Groq, APIStatusError' not in source:
    path.write_text(source.replace(marker,replacement))

# No trial resume, no old checkpoints, two fresh uncached dev trials only.
assert not (ROOT/'results/local/dev-groq-v2-flip-checkpoint.jsonl').exists()
import hashlib, base64, zipfile
protected=['agent_v2.py','safety.py','graders_v2.py','run.py','evals/tasks.jsonl','evals/holdout.lock.json','policy.md']
manifest={'source_commit':subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip(),'protocol':'35 dev tasks x 2 uncached trials; fresh; no resume; no old partial mixing','model':'openai/gpt-oss-120b','provider':'groq','agent':'v2','transport':'official Groq SDK defaults; only providers._post locally patched; daily quota stops','hashes':{p:hashlib.sha256((ROOT/p).read_bytes()).hexdigest() for p in protected},'cost_usd':0}
try:
    subprocess.run(['python3','run.py','--split','dev','--agent','groq','--model','openai/gpt-oss-120b','--agent-version','v2','--limit','35','--flip-check'],cwd=ROOT,check=True)
finally:
    out=ROOT/'results/local'
    out.mkdir(parents=True,exist_ok=True)
    (out/'fresh-manifest.json').write_text(json.dumps(manifest,indent=2))
    archive=shutil.make_archive('/content/fresh-flip-20261014','zip',out)
    print('FRESH_FLIP_ARCHIVE_BEGIN')
    print(base64.b64encode(Path(archive).read_bytes()).decode())
    print('FRESH_FLIP_ARCHIVE_END')
